# RAG System: Hardware Catalog Search

This notebook demonstrates a complete Retrieval-Augmented Generation (RAG) pipeline:

1. **Extract** text from PDF catalogs (Wurth Baer, Grass hinges & drawer slides)
2. **Chunk** documents into manageable pieces
3. **Embed** chunks using Ollama `nomic-embed-text` (local)
4. **Store** embeddings in ChromaDB vector database
5. **Query** with natural language, retrieving relevant context
6. **Generate** answers using both a local Ollama model and Anthropic Claude

## Prerequisites

- Ollama running locally with `nomic-embed-text` and a chat model (e.g. `mistral:7b`)
- Anthropic API key in `.env` file (copied from micro-x-agent-loop-python)
- PDF catalogs in the `catalogs/` folder

```bash
pip install -r requirements.txt
ollama pull nomic-embed-text
ollama pull mistral:7b
```

## 1. Setup & Configuration

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables from .env in the project root
load_dotenv(Path("../.env"))

# Configuration
CATALOG_DIR = Path("../catalogs")
CHROMA_PERSIST_DIR = Path("../chroma_db")
CHROMA_COLLECTION = "hardware_catalogs"

OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_EMBED_MODEL = "nomic-embed-text"
OLLAMA_CHAT_MODEL = "mistral:7b"  # Change to any Ollama model you have pulled

ANTHROPIC_MODEL = "claude-sonnet-4-20250514"

# Chunking parameters
CHUNK_SIZE = 1000      # characters per chunk
CHUNK_OVERLAP = 200    # overlap between chunks

print(f"Catalog directory: {CATALOG_DIR.resolve()}")
print(f"ChromaDB persist directory: {CHROMA_PERSIST_DIR.resolve()}")
print(f"Anthropic API key loaded: {'Yes' if os.getenv('ANTHROPIC_API_KEY') else 'No'}")
print(f"Ollama embed model: {OLLAMA_EMBED_MODEL}")
print(f"Ollama chat model: {OLLAMA_CHAT_MODEL}")

## 2. PDF Text Extraction

We use PyMuPDF (`fitz`) to extract text from each PDF page. Each page becomes a document with metadata tracking the source file and page number.

In [ ]:
import fitz  # PyMuPDF


def extract_pdf_pages(pdf_path: Path) -> list[dict]:
    """Extract text from each page of a PDF, returning a list of page documents."""
    documents = []
    doc = fitz.open(pdf_path)
    for page_num in range(len(doc)):
        page = doc[page_num]
        text = page.get_text().strip()
        if text:  # Skip blank pages
            documents.append({
                "text": text,
                "metadata": {
                    "source": pdf_path.name,
                    "page": page_num + 1,
                    "total_pages": len(doc),
                },
            })
    doc.close()
    return documents


# Extract text from all PDFs
all_pages = []
pdf_files = sorted(CATALOG_DIR.glob("*.pdf"))

for pdf_path in pdf_files:
    pages = extract_pdf_pages(pdf_path)
    all_pages.extend(pages)
    print(f"  {pdf_path.name}: {len(pages)} pages extracted")

print(f"\nTotal pages extracted: {len(all_pages)}")

## 3. Text Chunking

Large pages need to be split into smaller chunks that fit within embedding model context windows and provide focused retrieval results. We use a simple character-based chunking strategy with overlap to preserve context across chunk boundaries.

In [ ]:
def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    """Split text into overlapping chunks, breaking at sentence boundaries when possible."""
    if len(text) <= chunk_size:
        return [text]

    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size

        # Try to break at a sentence boundary (period, newline) near the end
        if end < len(text):
            # Look for a good break point in the last 20% of the chunk
            search_start = start + int(chunk_size * 0.8)
            last_period = text.rfind(".", search_start, end)
            last_newline = text.rfind("\n", search_start, end)
            break_point = max(last_period, last_newline)
            if break_point > search_start:
                end = break_point + 1

        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)

        start = end - overlap

    return chunks


# Chunk all extracted pages
all_chunks = []
for page_doc in all_pages:
    chunks = chunk_text(page_doc["text"])
    for i, chunk in enumerate(chunks):
        all_chunks.append({
            "text": chunk,
            "metadata": {
                **page_doc["metadata"],
                "chunk_index": i,
                "chunks_in_page": len(chunks),
            },
        })

print(f"Total chunks created: {len(all_chunks)}")
print(f"Average chunk length: {sum(len(c['text']) for c in all_chunks) / len(all_chunks):.0f} characters")

# Preview a chunk
print(f"\n--- Sample chunk (from {all_chunks[5]['metadata']['source']}, page {all_chunks[5]['metadata']['page']}) ---")
print(all_chunks[5]["text"][:300] + "...")

## 4. Ollama Embedding Function for ChromaDB

ChromaDB supports custom embedding functions. We create one that calls the local Ollama `/api/embed` endpoint with `nomic-embed-text`.

In [ ]:
import httpx
from chromadb.api.types import EmbeddingFunction, Documents, Embeddings


class OllamaEmbeddingFunction(EmbeddingFunction):
    """ChromaDB-compatible embedding function using Ollama."""

    def __init__(self, model: str = OLLAMA_EMBED_MODEL, base_url: str = OLLAMA_BASE_URL):
        self.model = model
        self.base_url = base_url

    def __call__(self, input: Documents) -> Embeddings:
        embeddings = []
        # Process in batches to avoid overwhelming Ollama
        batch_size = 32
        for i in range(0, len(input), batch_size):
            batch = input[i : i + batch_size]
            response = httpx.post(
                f"{self.base_url}/api/embed",
                json={"model": self.model, "input": batch},
                timeout=120.0,
            )
            response.raise_for_status()
            data = response.json()
            embeddings.extend(data["embeddings"])
        return embeddings


# Test the embedding function
embed_fn = OllamaEmbeddingFunction()
test_embedding = embed_fn(["test sentence about cabinet hinges"])
print(f"Embedding dimension: {len(test_embedding[0])}")
print(f"First 5 values: {test_embedding[0][:5]}")

## 5. Build the ChromaDB Vector Store

We create a persistent ChromaDB collection and add all document chunks with their embeddings and metadata.

In [ ]:
import chromadb

# Create persistent ChromaDB client
chroma_client = chromadb.PersistentClient(path=str(CHROMA_PERSIST_DIR.resolve()))

# Delete existing collection if it exists (for clean re-runs)
try:
    chroma_client.delete_collection(CHROMA_COLLECTION)
    print(f"Deleted existing collection '{CHROMA_COLLECTION}'")
except ValueError:
    pass

# Create collection with our Ollama embedding function
collection = chroma_client.create_collection(
    name=CHROMA_COLLECTION,
    embedding_function=embed_fn,
    metadata={"hnsw:space": "cosine"},  # Use cosine similarity
)

print(f"Created collection '{CHROMA_COLLECTION}'")
print(f"Adding {len(all_chunks)} chunks...")

In [ ]:
# Add chunks in batches (ChromaDB + Ollama embedding can be slow for large batches)
BATCH_SIZE = 50

for i in range(0, len(all_chunks), BATCH_SIZE):
    batch = all_chunks[i : i + BATCH_SIZE]
    collection.add(
        ids=[f"chunk_{i + j}" for j in range(len(batch))],
        documents=[c["text"] for c in batch],
        metadatas=[c["metadata"] for c in batch],
    )
    progress = min(i + BATCH_SIZE, len(all_chunks))
    print(f"  Added {progress}/{len(all_chunks)} chunks", end="\r")

print(f"\nDone! Collection now has {collection.count()} documents.")

## 6. Retrieval: Semantic Search

Now we can query the vector store with natural language. ChromaDB handles embedding the query and finding the most similar chunks.

In [ ]:
def retrieve(query: str, n_results: int = 5) -> list[dict]:
    """Retrieve the most relevant chunks for a query."""
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
    )

    retrieved = []
    for i in range(len(results["ids"][0])):
        retrieved.append({
            "id": results["ids"][0][i],
            "text": results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            "distance": results["distances"][0][i],
        })
    return retrieved


# Test retrieval
query = "What types of concealed hinges are available?"
results = retrieve(query)

print(f"Query: {query}\n")
for i, r in enumerate(results):
    print(f"Result {i+1} (distance: {r['distance']:.4f})")
    print(f"  Source: {r['metadata']['source']}, Page {r['metadata']['page']}")
    print(f"  Text: {r['text'][:150]}...\n")

## 7. RAG Generation: Build the Full Pipeline

We combine retrieval with LLM generation. The retrieved chunks become context in the prompt, and the LLM synthesizes an answer.

In [ ]:
def build_rag_prompt(query: str, context_chunks: list[dict]) -> str:
    """Build a RAG prompt with retrieved context."""
    context_parts = []
    for i, chunk in enumerate(context_chunks, 1):
        source = chunk["metadata"]["source"]
        page = chunk["metadata"]["page"]
        context_parts.append(f"[Source {i}: {source}, Page {page}]\n{chunk['text']}")

    context_text = "\n\n---\n\n".join(context_parts)

    return f"""You are a knowledgeable hardware product specialist. Answer the user's question
based ONLY on the provided catalog context. If the context doesn't contain enough
information to fully answer the question, say so clearly.

When referencing specific products or specifications, cite the source catalog and page number.

CONTEXT FROM PRODUCT CATALOGS:
{context_text}

USER QUESTION: {query}

ANSWER:"""

### 7a. Generate with Ollama (Local)

In [ ]:
from openai import OpenAI

# Ollama exposes an OpenAI-compatible API
ollama_client = OpenAI(
    base_url=f"{OLLAMA_BASE_URL}/v1",
    api_key="ollama",  # Ollama doesn't need a real key
)


def rag_query_ollama(query: str, n_results: int = 5) -> str:
    """Full RAG pipeline using Ollama for generation."""
    # Retrieve
    chunks = retrieve(query, n_results=n_results)

    # Generate
    prompt = build_rag_prompt(query, chunks)
    response = ollama_client.chat.completions.create(
        model=OLLAMA_CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
    )

    return response.choices[0].message.content


# Test it
query = "What types of concealed hinges are available and what are their opening angles?"
print(f"Query: {query}\n")
print("=" * 60)
print("OLLAMA RESPONSE:")
print("=" * 60)
answer = rag_query_ollama(query)
print(answer)

### 7b. Generate with Anthropic Claude

In [ ]:
import anthropic

claude_client = anthropic.Anthropic()  # Uses ANTHROPIC_API_KEY from environment


def rag_query_claude(query: str, n_results: int = 5) -> str:
    """Full RAG pipeline using Anthropic Claude for generation."""
    # Retrieve
    chunks = retrieve(query, n_results=n_results)

    # Generate
    prompt = build_rag_prompt(query, chunks)
    response = claude_client.messages.create(
        model=ANTHROPIC_MODEL,
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}],
    )

    return response.content[0].text


# Test it
query = "What types of concealed hinges are available and what are their opening angles?"
print(f"Query: {query}\n")
print("=" * 60)
print("CLAUDE RESPONSE:")
print("=" * 60)
answer = rag_query_claude(query)
print(answer)

## 8. Compare Both Models Side-by-Side

Let's run several queries through both models to compare answer quality.

In [ ]:
test_queries = [
    "What drawer slide options are available for full extension?",
    "What is the weight capacity of the Nexis drawer system?",
    "Compare the Tiomos and Nexis product lines.",
    "What soft-close options are available for cabinet hinges?",
]

for query in test_queries:
    print("\n" + "#" * 70)
    print(f"QUERY: {query}")
    print("#" * 70)

    # Retrieve once, show sources
    chunks = retrieve(query)
    print(f"\nRetrieved {len(chunks)} chunks from:")
    for c in chunks:
        print(f"  - {c['metadata']['source']}, Page {c['metadata']['page']} (distance: {c['distance']:.4f})")

    # Ollama
    prompt = build_rag_prompt(query, chunks)

    print(f"\n--- Ollama ({OLLAMA_CHAT_MODEL}) ---")
    ollama_resp = ollama_client.chat.completions.create(
        model=OLLAMA_CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
    )
    print(ollama_resp.choices[0].message.content)

    # Claude
    print(f"\n--- Claude ({ANTHROPIC_MODEL}) ---")
    claude_resp = claude_client.messages.create(
        model=ANTHROPIC_MODEL,
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}],
    )
    print(claude_resp.content[0].text)

    print()

## 9. Interactive Query

Use this cell to run your own queries against the catalog knowledge base.

In [ ]:
# Change this query to ask anything about the hardware catalogs
my_query = "What mounting options exist for face frame cabinets?"

# Choose provider: "ollama" or "claude"
provider = "claude"

print(f"Query: {my_query}")
print(f"Provider: {provider}\n")

# Retrieve and display sources
chunks = retrieve(my_query, n_results=5)
print("Retrieved sources:")
for c in chunks:
    print(f"  - {c['metadata']['source']}, Page {c['metadata']['page']} (distance: {c['distance']:.4f})")
print()

# Generate answer
if provider == "ollama":
    answer = rag_query_ollama(my_query)
else:
    answer = rag_query_claude(my_query)

print(answer)

## 10. Inspect the Vector Store

Useful utilities for exploring what's in the ChromaDB collection.

In [ ]:
# Collection stats
print(f"Collection: {CHROMA_COLLECTION}")
print(f"Total documents: {collection.count()}")

# Documents per source
all_metadata = collection.get(include=["metadatas"])["metadatas"]
source_counts = {}
for m in all_metadata:
    src = m["source"]
    source_counts[src] = source_counts.get(src, 0) + 1

print("\nChunks per source:")
for src, count in sorted(source_counts.items()):
    print(f"  {src}: {count} chunks")

In [ ]:
# Reload an existing collection (useful when restarting the notebook
# without re-ingesting all the PDFs)

# chroma_client = chromadb.PersistentClient(path=str(CHROMA_PERSIST_DIR.resolve()))
# embed_fn = OllamaEmbeddingFunction()
# collection = chroma_client.get_collection(
#     name=CHROMA_COLLECTION,
#     embedding_function=embed_fn,
# )
# print(f"Loaded collection with {collection.count()} documents")